# Tokenization & Embeddings: Text to Vectors

Reach for this when you need: 
- Reference for HuggingFace `AutoTokenizer` and PyTorch `nn.Embedding`.
- To understand BPE, WordPiece, and subword tokenization.
- Implementation for padding and attention masking.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. HuggingFace AutoTokenizer
Modern NLP uses subword tokenizers (BPE for GPT, WordPiece for BERT) to handle out-of-vocabulary words.

| Method | Action | Purpose |
| :--- | :--- | :--- |
| `encode` | Map text to IDs | Input for models |
| `decode` | Map IDs back to text | Human-readable output |
| `__call__` | Returns IDs + Masks | Standard way to prepare batch |
| `pad` | Normalize seq lengths | Ensuring uniform batch shapes |

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "PyTorch is incredible for NLP."

# Preferred call: returns input_ids and attention_mask
inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

print(f"Input IDs: {inputs['input_ids']}") # [1, 10]
print(f"Attention Mask: {inputs['attention_mask']}")

## 2. PyTorch nn.Embedding

A lookup table that maps integer indices to dense vectors. 

✅ **Use when**: Implementing a custom encoder/decoder from scratch.
❌ **Don't use when**: You are fine-tuning a BERT/GPT model (use the model's existing embedding layer).

In [ ]:
# nn.Embedding(vocab_size, embedding_dim)
embedding = nn.Embedding(num_embeddings=30522, embedding_dim=768)

input_ids = torch.tensor([[101, 200, 102]]) # Batch of 1 sequence
embedded_vectors = embedding(input_ids)

print(f"Embedding shape: {embedded_vectors.shape}") # [1, 3, 768]

### Common Pitfalls
- **Vocab Mismatch**: Using a BERT tokenizer with a GPT model will produce garbage results. ALWAYS use the tokenizer named after the model weights.
- **Special Tokens**: Forgetting that BERT needs `[CLS]` and `[SEP]` tokens; `AutoTokenizer` adds these automatically in `__call__`.
- **Truncation**: If sequence length exceeds max context (e.g. 512 for BERT), models will crash unless `truncation=True` is set.

### Key Takeaways
- `input_ids` are integers; `embeddings` are vectors. This is the bridge between raw text and math.
- `attention_mask` (1s for real tokens, 0s for padding) is critical for self-attention layers to ignore padding.
- Subword tokenization solves the 'unknown word' problem by breaking words into meaningful chunks (e.g., "incredible" -> "in", "##credible").